In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH = 50
DIMENSIONAL_CONSTRAINT = 0.8
RECURSION_DEPTHS = list(range(10, MAX_RECURSION_DEPTH + 1, 10))  # [10, 20, 30, 40, 50]

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=1.0):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth, body_fn, x)

# --- Sharding Setup (Fixed for TPU v5e-1, which has only 1 device)
devices = jax.devices()  # Automatically detect TPU devices
sharding = PositionalSharding(devices)  # Adjust sharding dynamically

batch_size = 50_000
data_size = 50_000
batch_input = jnp.linspace(0, 10, data_size)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(
        lambda xi: dppu_with_dynamic_pi_phi(xi, depth=10, scale_factor=0.5), in_axes=0
    )(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# --- 1️⃣ Optimize TPU Workloads By Splitting Computations into Depth=10 Segments
def split_into_depth_10(x, total_depth):
    """Runs dppu_with_dynamic_pi_phi in Depth=10 chunks instead of deeper recursion."""
    iterations = total_depth // 10
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=10)
    return x

# --- Run with Depth Intervals of 10
for depth in RECURSION_DEPTHS:
    if depth == 10:
        output_batch = batched_dppu_processing(batch_input)
    else:
        output_batch = split_into_depth_10(batch_input, depth)

    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)

NUM_TRIALS = 10
INPUT_SIZE = 50_000

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)

for depth in RECURSION_DEPTHS:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        if depth == 10:
            result = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)
        else:
            result = split_into_depth_10(jnp.ones((INPUT_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Size={INPUT_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

jax.devices()

# --- 2️⃣ TPU Benchmarking: Compare Across TPU Models
# This part should be executed manually on TPU v4, v5e, and v5p for comparison
print("\n🚀 TPU Model:", jax.devices()[0].device_kind)

# --- 3️⃣ Investigate Why TPU Compiles Depth=10 Differently with XLA Debugging
compiled_fn_10 = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((50_000,)), depth=10)
compiled_fn_20 = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((50_000,)), depth=20)

print("\n🚀 XLA Compilation for Depth=10:")
print(compiled_fn_10.as_text())

print("\n🚀 XLA Compilation for Depth=20:")
print(compiled_fn_20.as_text())

Batch Output Shape (Depth=10): (50000,)
Batch Output Shape (Depth=20): (50000,)
Batch Output Shape (Depth=30): (50000,)
Batch Output Shape (Depth=40): (50000,)
Batch Output Shape (Depth=50): (50000,)

🔥 TPU Benchmark (Depth=10, Size=50000)
Avg: 0.001365, Min: 0.001258, Max: 0.001773

🔥 TPU Benchmark (Depth=20, Size=50000)
Avg: 0.001325, Min: 0.001273, Max: 0.001498

🔥 TPU Benchmark (Depth=30, Size=50000)
Avg: 0.001369, Min: 0.001301, Max: 0.001440

🔥 TPU Benchmark (Depth=40, Size=50000)
Avg: 0.001408, Min: 0.001372, Max: 0.001475

🔥 TPU Benchmark (Depth=50, Size=50000)
Avg: 0.001424, Min: 0.001379, Max: 0.001462

🚀 TPU Model: TPU v5 lite

🚀 XLA Compilation for Depth=10:
module @jit_dppu_with_dynamic_pi_phi attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<50000xf32> {mhlo.layout_mode = "default"}, %arg1: tensor<i32> {mhlo.layout_mode = "default"}) -> (tensor<50000xf32> {jax.result_info = "", mhlo.layout_mode = "default"}) {

/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
